### 1. Basic Tasks

In [0]:
-- 1. Use CTAS with read_files() to ingest a CSV file into a managed Delta table.
create table if not exists cyntexa_dev.sales.sales_raw as
select * from read_files('/Volumes/cyntexa_dev/sales/raw/sales.csv', header => 'true')

In [0]:
select * from cyntexa_dev.sales.sales_raw

In [0]:
-- 2. Ingest a nested JSON file, extracting at least 2 nested fields into top-level columns.
create table cyntexa_dev.sales.driver_raw as
select code, dob, driverId, driverRef, name.forename as first_name, name.surname as last_name, nationality, number , url from read_files('/Volumes/cyntexa_dev/sales/raw/drivers.json')

In [0]:
-- 3. Run DESCRIBE, DESCRIBE EXTENDED, and DESCRIBE DETAIL on your new table and note what unique information each one gives you.
describe cyntexa_dev.sales.sales_raw

In [0]:
describe extended cyntexa_dev.sales.sales_raw

In [0]:
describe detail cyntexa_dev.sales.sales_raw

differences between 3 describe commands

### 2. Intermediate Tasks

In [0]:
-- 4. Add _metadata.file_name and _metadata.file_path to your ingestion query and use them to prove which source file each row came from.
drop table if exists cyntexa_dev.sales.sales_raw;

create table if not exists cyntexa_dev.sales.sales_raw as
select *, _metadata.file_name as file_name, _metadata.file_path as file_path from read_files('/Volumes/cyntexa_dev/sales/raw/sales.csv', header => 'true')


In [0]:
-- Create an Iceberg table from the same source data and compare its DESCRIBE DETAIL output (format, location) to the Delta version.
create table if not exists cyntexa_dev.sales.sales_raw_iceberg using iceberg as
select * from read_files('/Volumes/cyntexa_dev/sales/raw/sales.csv', header => 'true') 

In [0]:
describe detail cyntexa_dev.sales.sales_raw_iceberg

In [0]:
-- 6. (Data Analyst) Write a query using the metadata columns to build a 'records per source file' audit report — useful for verifying a vendor's daily file drop.
select file_name, count(*) as records_per_file 
from cyntexa_dev.sales.sales_raw
group by file_name 


### 3. Advanced Tasks

### 7.

In [0]:
select * from read_files("/Volumes/cyntexa_dev/sales/raw/customers/customers1.csv",
format => 'csv',
header => true,
sep => ',')

In [0]:
-- Reading files with extra column stored in rescued data 
select * from read_files("/Volumes/cyntexa_dev/sales/raw/customers/customers2.csv",
format => 'csv',
header => true,
sep => ',',
schema => 'customer_id int, name string, email string',
rescuedDataColumn => '_rescued_data')

In [0]:
-- check if any data was rescued 

select * from read_files("/Volumes/cyntexa_dev/sales/raw/customers/customers2.csv",
format => 'csv',
header => true,
sep => ',',
schema => 'customer_id int, name string, email string',
rescuedDataColumn => '_rescued_data')
where _rescued_data is not null

In [0]:
-- Columns interpreted incorrectly due to mismatched seperator 

select * from read_files("/Volumes/cyntexa_dev/sales/raw/customers/customers3.csv",
format => 'csv',
header => true,
sep => ',') 

In [0]:
select * from read_files("/Volumes/cyntexa_dev/sales/raw/customers/customers3.csv",
format => 'csv',
header => true,
sep => '|')

### 8. 
 
An organization should generally use native Delta Lake when a table is primarily being managed and used by Databricks. However Iceberg or Delta UniForm should be considered when the same table needs to be accessed by multiple query engines like Snowflake 
 
- Native Delta is preferred choice when Databricks is the primary platform for the table. It provides strong integration with ACID transactions, Time Travel, schema enforcement and pipelines. 
 
- Iceberg is better choice when open table format interoperability is an important requirement. If same data needs to be accessed by different engines such as Snowflakes or other platforms that have strong Iceberg support, Iceberg can reduce platform specific dependency.  
 
- When an organization wants to keep Delta as the primary table format while improving interoperability with engines that support Iceberg or other open table formats, this can be achieved using Delta UniForm. It allows a Delta table to be read by engines that understand supported open formats without requiring the organization to maintain separate copies

### 9.

In [0]:
create table cyntexa_dev.sales.customers as 
select *, _metadata.file_name as file_name,
_metadata.file_path as file_path
from read_files("/Volumes/cyntexa_dev/sales/raw/customers/customers1.csv",
format => 'csv',
header => true,
inferSchema => true)

In [0]:
desc history cyntexa_dev.sales.customers

In [0]:
insert into cyntexa_dev.sales.customers
select *, _metadata.file_name as file_name,
_metadata.file_path as file_path 
from read_files("/Volumes/cyntexa_dev/sales/raw/customers/customers4.csv",
format => 'csv',
header => true,
inferSchema => true)

In [0]:
desc history cyntexa_dev.sales.customers

In [0]:
-- bad data at id 105
select * from cyntexa_dev.sales.customers where customer_id = 105